# 17 — Neighborhood-size diagnostics for representation geometry (Phi-2)

**Research question:** why does increasing k from 10 to 20 reduce the late-layer `pre_last` correctness gap while the `finals` gap remains approximately stable?

This notebook diagnoses an existing observation; it is **exploratory**, not a fresh confirmatory test. It reuses the exact common_clean cohort, original matched plans, extracted representations and estimator from FINAL notebook 13. No Phi-2 generation or extraction is needed. Notebook 16's probes are not used here.

## Fixed protocol

- Views: `finals` (last stored full-sequence token, not necessarily the answer token) and `pre_last` (last token before the selected numerical span, which may follow earlier answer mentions).
- All 32 layers in FINAL. Existing L24–L31 summary retained; layer averages are calculated within draw before sampling percentiles. QUICK uses L0, L24, L31 only and produces **no** L24–L31 summary.
- k = **5, 8, 10, 15, 20, 30, 40**. The k=5/8 values are numerical/small-neighborhood diagnostics, not a new primary result. No best-k selection.
- Same pooled, within-problem matched plans as notebook 13: 120 draws, 397 points per class in the full cohort. This is pooled geometry, not an average of per-problem ID estimates.
- Additional nested caps 3 and 1 retain all 129 problems. They use the first selected attempts in each original random per-problem draw; the full cap-10 plan remains exactly unchanged. Caps affect both N and within-problem multiplicity and are not a pure sample-size intervention.
- Float64 row-L2 normalization, direct Euclidean distances, original **mean of local inverse estimates** with k-1 denominator. No coordinate standardization, epsilon clipping, jitter, new filtering or bootstrap-with-replacement.
- Before analysing a layer, recompute the original k=10/20 class means and Delta ID using **all 120 original plans**, including in QUICK, and compare to FINAL 13. Stop if the reference is not reproduced.

## Outputs and interpretation

1. Separate correct-ID, incorrect-ID and Delta-ID curves by k and layer, plus paired k and representation contrasts.
2. Neighbor problem identities, same-problem fractions, distance/radius summaries and distance-tie diagnostics.
3. Local-ID quantiles, upper-tail shares, and paired per-point changes from k=10 to 20.
4. Nested sampling-cap curves and paired contrasts.

Summary statistics use every requested draw. **Detailed raw neighbor records are stored for draw 0 only by default**, for every computed layer/view/cap/class. This fixed choice prevents cherry-picking and excessive storage. Change DETAIL_DRAWS before running if more records are needed; that creates a separate run. The 41st neighbor is retained to audit ties at the k=40 boundary.

Sampling percentiles are descriptive ranges, **not confidence intervals**. Repeated draws reuse attempts; rows are not independent replications. Same k does not imply the same distance scale across representations or classes. Problem identity is a diagnostic of neighborhood composition, not a causal explanation. No new permutation or LOPO inference is performed in this notebook; earlier LOPO results remain separate.

## How to run in Colab

1. Upload this notebook and select a GPU runtime if available. CPU works but distance calculation can be slow.
2. Keep `QUICK=True` and run cells **1–7** in order. Existing FINAL 13 artifacts and representation caches must be in Drive. If automatic source discovery fails, set `SOURCE13` in cell 1 to the directory shown in your earlier FINAL 13 run, or an extracted aligned review bundle.
3. Inspect reproduction checks and send the QUICK review bundle for an execution review.
4. Then set `QUICK=False` in cell 1 and run **1–7** again for FINAL. QUICK and FINAL use separate directories.

You do not need to rerun notebooks 13, 15 or 16. The new distance calculations are required because earlier compact summaries do not contain full neighbor geometry. Each completed representation/layer is checkpointed. After a restart, run cells 1–7 in order with the same configuration; completed layers are verified and skipped. Changing the device or software versions creates a separate run signature.

Only load your own trusted `.pt` extraction files. The notebook does not install packages; standard Colab NumPy, pandas, torch, matplotlib and tqdm are sufficient.

In [ ]:
# 1 — Configuration. Run all cells in order. QUICK is an execution test, not final evidence.
import json, hashlib, platform, gc, shutil, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/msc_thesis')
SOURCE13 = None  # Optional: exact FINAL 13 directory, or extracted aligned_final_review_bundle.
QUICK = True
RUN_DIAGNOSTICS = True  # False runs the input/plan audit only; cells 5–7 then stop explicitly.
K_VALUES = [5, 8, 10, 15, 20, 30, 40]
VIEWS = ['finals', 'pre_last']
CAPS = [10, 3, 1]  # Full original plan, then nested within-problem subsets.
LAYERS = [0, 24, 31] if QUICK else list(range(32))
DRAWS = 3 if QUICK else 120
DETAIL_DRAWS = [0]  # Fixed in advance. Full neighbor/local records for these draws only.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EXPECTED_SOURCE_SHA = '2bb21797cc39443a066b47a9afab2bece98d99e08879562d05ca0d0c7df08e74'
ORIGINAL = BASE/'data/representations/phi2/phi2_repr_paired_v1.pt'
PREFIX = BASE/'data/representations/phi2/phi2_preanswer_paired_v1.pt'
CONFIG = dict(version='neighborhood_diagnostic_v1', quick=QUICK, k_values=K_VALUES,
    views=VIEWS, layers=LAYERS, draws=DRAWS, caps=CAPS, detail_draws=DETAIL_DRAWS,
    normalize=True, dtype='float64', distance='direct cdist, no matmul',
    estimator='mean of local inverses, k-1 denominator; identical to notebook 13',
    tie_rule='stable original plan order', reference_reps=120,
    late_window=list(range(24,32)), source_manifest_sha256=EXPECTED_SOURCE_SHA,
    implementation_sha256='63f98ad4f4a49be1d610769f4c0097bf1c98c825fd3d458554c5be364a6e0e1a')

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()

def bool_values(values):
    a = pd.Series(values).astype(str).str.lower()
    if not a.isin(['true','false','0','1']).all(): raise ValueError('Invalid Boolean values')
    return a.isin(['true','1']).to_numpy()

print('Mode:', 'QUICK' if QUICK else 'FINAL', '| device:', DEVICE)
print('Layers:', LAYERS, '| draws:', DRAWS, '| fixed k grid:', K_VALUES)


Functions preserve the original estimator and explicitly reject degenerate neighborhoods. Stable distance ties use original plan order.

In [ ]:
# 2 — Numerical and sampling functions.
"""Neighborhood diagnostics. NumPy only; no fitting, jitter, clipping or filtering."""
import hashlib
import numpy as np


def normalize_rows(x):
    x = np.asarray(x, np.float64)
    if x.ndim != 2 or not np.isfinite(x).all():
        raise ValueError('Expected finite [N,D] vectors')
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    if (norms <= 1e-12).any():
        raise FloatingPointError('Zero/near-zero input norm')
    return x / norms


def local_lb(distances, k):
    a = np.asarray(distances, np.float64)[..., :k]
    if k < 2 or a.shape[-1] != k or not np.isfinite(a).all() or (a <= 1e-12).any():
        raise FloatingPointError(f'Invalid neighbors at k={k}; no points removed')
    denominator = np.log(a[..., -1:] / a[..., :-1]).mean(axis=-1)
    if not np.isfinite(denominator).all() or (denominator <= 1e-12).any():
        raise FloatingPointError(f'Degenerate LB denominator at k={k}; no clipping')
    local = 1.0 / denominator
    if not np.isfinite(local).all():
        raise FloatingPointError('Nonfinite local ID')
    return local, denominator


def validate_plans(plans, pids, labels, cap=10):
    plans = np.asarray(plans)
    if plans.ndim != 3 or plans.shape[1] != 2 or plans.dtype.kind not in 'iu':
        raise ValueError('Plans must be integer [draw,2,N]')
    if plans.min() < 0 or plans.max() >= len(pids):
        raise ValueError('Plan index out of cohort bounds')
    for C, I in plans:
        if len(np.unique(np.r_[C, I])) != 2 * len(C):
            raise ValueError('Repeated attempt in matched draw')
        if not labels[C].all() or labels[I].any():
            raise ValueError('Wrong class in matched plan')
        if not np.array_equal(pids[C], pids[I]):
            raise ValueError('Unmatched problem identities/order')
        ids, counts = np.unique(pids[C], return_counts=True)
        expected = np.asarray([min(cap, np.sum((pids == p) & labels), np.sum((pids == p) & ~labels)) for p in ids])
        if not np.array_equal(ids, np.unique(pids)) or not np.array_equal(counts, expected):
            raise ValueError('Unexpected problem support or counts')


def regenerated_plans(pids, labels, cap, reps, seed):
    # Reproduce notebook 13 RNG calls and original attempt order exactly.
    rng = np.random.default_rng(seed)
    result = []
    for _ in range(reps):
        C, I = [], []
        for pid in np.unique(pids):
            c = np.flatnonzero((pids == pid) & labels)
            i = np.flatnonzero((pids == pid) & ~labels)
            n = min(cap, len(c), len(i))
            C.extend(rng.choice(c, n, replace=False))
            I.extend(rng.choice(i, n, replace=False))
        result.append([C, I])
    return np.asarray(result, dtype=np.int64)


def nested_plans(full, pids, cap):
    # Random order comes from the original draw, shared across all representations/layers/k.
    result = []
    for C, I in full:
        keep = np.zeros(len(C), bool)
        for pid in np.unique(pids[C]):
            keep[np.flatnonzero(pids[C] == pid)[:cap]] = True
        result.append([C[keep], I[keep]])
    return np.asarray(result, dtype=np.int64)


def neighbors(D, indices, max_k):
    indices = np.asarray(indices, int)
    if len(indices) <= max_k + 1:
        raise ValueError('Need N > max_k+1 to audit boundary ties')
    sub = np.asarray(D[np.ix_(indices, indices)], np.float64).copy()
    np.fill_diagonal(sub, np.inf)
    # Stable sort fixes equal-distance ties by original plan position, never label or ID estimate.
    pos = np.argsort(sub, axis=1, kind='stable')[:, :max_k + 1]
    d = np.take_along_axis(sub, pos, axis=1)
    if not np.isfinite(d).all() or (d <= 1e-12).any():
        raise FloatingPointError('Nonpositive/invalid neighbor distance; inspect source, do not discard')
    return indices[pos], d


def diagnose_class(D, indices, pids, k_values):
    indices = np.asarray(indices, int)
    nn, d = neighbors(D, indices, max(k_values))
    same = pids[nn] == pids[indices, None]
    n = len(indices)
    counts = {p: int(np.sum(pids[indices] == p)) for p in np.unique(pids[indices])}
    expected = np.asarray([(counts[p] - 1) / (n - 1) for p in pids[indices]])
    all_local, all_den, rows = [], [], []
    tail_n = max(1, int(np.ceil(.05 * n)))
    for k in k_values:
        local, den = local_lb(d, k)
        all_local.append(local); all_den.append(den)
        own = same[:, :k].mean(axis=1)
        q = np.quantile(local, [.05, .25, .5, .75, .9, .95, .99])
        top = np.sort(local)[-tail_n:]
        rows.append(dict(k=int(k), n=int(n), id_mean=float(local.mean()),
            local_min=float(local.min()), local_p05=q[0], local_p25=q[1], local_median=q[2],
            local_p75=q[3], local_p90=q[4], local_p95=q[5], local_p99=q[6], local_max=float(local.max()),
            top5_share=float(top.sum()/local.sum()), top5_n=tail_n,
            denominator_min=float(den.min()), denominator_p01=float(np.quantile(den,.01)),
            nearest_min=float(d[:,0].min()), radius_mean=float(d[:,k-1].mean()),
            radius_median=float(np.median(d[:,k-1])), radius_p90=float(np.quantile(d[:,k-1],.9)),
            own_fraction=float(own.mean()), own_expected=float(expected.mean()),
            own_excess=float((own-expected).mean()),
            adjacent_tie_fraction=float((np.diff(d[:,:k],axis=1)==0).mean()),
            boundary_tie_fraction=float((d[:,k-1]==d[:,k]).mean())))
    rank_rows = [dict(rank=j+1, distance_mean=float(d[:,j].mean()),
        distance_median=float(np.median(d[:,j])), own_fraction=float(same[:,j].mean()),
        own_expected=float(expected.mean())) for j in range(max(k_values))]
    local = np.stack(all_local, axis=1)
    shift = local[:,k_values.index(20)]-local[:,k_values.index(10)]
    top_abs = np.argsort(np.abs(shift), kind='stable')[-tail_n:]
    total_abs = np.abs(shift).sum()
    shift_row = dict(n=n, shift_mean=float(shift.mean()), shift_median=float(np.median(shift)),
        shift_p05=float(np.quantile(shift,.05)), shift_p95=float(np.quantile(shift,.95)),
        fraction_negative=float((shift<0).mean()), fraction_positive=float((shift>0).mean()),
        positive_contribution=float(np.maximum(shift,0).sum()/n),
        negative_contribution=float(np.minimum(shift,0).sum()/n),
        top_abs5_contribution=float(shift[top_abs].sum()/n),
        remainder_contribution=float((shift.sum()-shift[top_abs].sum())/n),
        top_abs5_share=float(np.abs(shift[top_abs]).sum()/total_abs) if total_abs else 0.0,
        own_first10=float(same[:,:10].mean()), own_next10=float(same[:,10:20].mean()),
        radius20_over10_median=float(np.median(d[:,19]/d[:,9])))
    detail = dict(query_indices=indices, neighbor_indices=nn, distances=d,
                  local_id=local, denominators=np.stack(all_den,axis=1), own_problem=same)
    return rows, rank_rows, shift_row, detail


def baseline_ids(D, plans, k_values=(10,20)):
    result = np.empty((len(plans),2,len(k_values)))
    for r, plan in enumerate(plans):
        for side, ids in enumerate(plan):
            # Mirrors notebook 13: argpartition then sorted distances.
            sub = np.asarray(D[np.ix_(ids, ids)],np.float64).copy()
            np.fill_diagonal(sub,np.inf)
            d = np.partition(sub,max(k_values)-1,axis=1)[:,:max(k_values)]
            d.sort(axis=1)
            for j,k in enumerate(k_values): result[r,side,j] = local_lb(d,k)[0].mean()
    return result


def atomic_npz(path, **arrays):
    temp = path.with_suffix('.tmp')
    with temp.open('wb') as f: np.savez_compressed(f,**arrays)
    temp.replace(path)


def read_signed_npz(path, signature):
    with np.load(path,allow_pickle=False) as z:
        if str(z['signature'].item()) != signature:
            raise ValueError(f'Stale checkpoint: {path}')
        return {k:z[k].copy() for k in z.files if k!='signature'}


Source manifest and both representation caches are hash-checked. Saved plans are also reconstructed from the original seed and compared exactly.

In [ ]:
# 3 — Verify original cohort, cache provenance, and all 120 original plans.
if SOURCE13 is None:
    candidates = sorted((BASE/'results/aligned_four_views_phi2_v1/final').glob('*/manifest.json'))
    matches = [p.parent for p in candidates if digest(p)==EXPECTED_SOURCE_SHA]
    if len(matches)!=1:
        raise ValueError('Set SOURCE13 to the original FINAL 13 folder (or its extracted review bundle). '
                         'Candidates: '+str([str(p.parent) for p in candidates]))
    SOURCE13 = matches[0]
SOURCE13 = Path(SOURCE13)
if digest(SOURCE13/'manifest.json') != EXPECTED_SOURCE_SHA:
    raise ValueError('Source manifest differs from the aligned analysis being diagnosed')
SOURCE_MANIFEST = json.loads((SOURCE13/'manifest.json').read_text())
sc = SOURCE_MANIFEST['config']
assert sc['quick'] is False and sc['pooled_reps']==120 and sc['cap']==10 and sc['normalize'] is True
assert sc['k_values']==[10,20] and sc['layers']==list(range(32))
for path in [ORIGINAL,PREFIX]:
    expected = [v for k,v in SOURCE_MANIFEST['inputs'].items() if Path(k).name==path.name]
    if len(expected)!=1 or digest(path)!=expected[0]:
        raise ValueError(f'Representation cache differs from FINAL 13: {path}')

all_meta = pd.read_csv(SOURCE13/'cohort_membership.csv')
assert not all_meta.duplicated(['problem_id','attempt_id']).any()
all_keys = all_meta[['problem_id','attempt_id']].to_numpy(int)
assert np.array_equal(all_keys,np.asarray(SOURCE_MANIFEST['common_keys'],int))
clean_indices = np.asarray(SOURCE_MANIFEST['clean_indices'],int)
assert np.array_equal(clean_indices,np.flatnonzero(~bool_values(all_meta.excluded)))
META = all_meta.iloc[clean_indices].reset_index(drop=True).copy()
META['correct'] = bool_values(META.correct)
KEYS = META[['problem_id','attempt_id']].to_numpy(int)
PIDS = META.problem_id.to_numpy(int)
LABELS = META.correct.to_numpy(bool)
with np.load(SOURCE13/'common_clean_plans.npz',allow_pickle=False) as z:
    FULL_PLANS = z['pooled'].copy()
    assert np.array_equal(z['cohort_keys'],KEYS)
validate_plans(FULL_PLANS,PIDS,LABELS,cap=10)
assert len(FULL_PLANS)==120
expected = regenerated_plans(PIDS,LABELS,10,120,sc['matching_seed'])
assert np.array_equal(FULL_PLANS,expected), 'Saved plans do not match original sampling protocol'
del expected
REFERENCE = pd.read_csv(SOURCE13/'aligned_id_summary.csv')
REFERENCE = REFERENCE[REFERENCE.cohort=='common_clean'].copy()
assert not REFERENCE.duplicated(['representation','layer','k']).any()
for v in VIEWS:
    q = REFERENCE[REFERENCE.representation==v]
    assert len(q)==64 and set(q.k)=={10,20} and set(q.layer)==set(range(32))
    assert (q.matched_per_class==FULL_PLANS.shape[-1]).all() and (q.reps_pooled==120).all()
assert len(META)==2541 and len(np.unique(PIDS))==129 and LABELS.sum()==433
print('Verified source:',SOURCE13)
print('Cohort:',len(META),'attempts,',len(np.unique(PIDS)),'problems')
print('Original plans:',FULL_PLANS.shape,'| preserved exactly')


This audit freezes the protocol before new results. Set RUN_DIAGNOSTICS=False in cell 1 if you want to stop here first.

In [ ]:
# 4 — Freeze nested plans and output signature. No ID calculation yet.
PLANS = {cap:nested_plans(FULL_PLANS[:DRAWS],PIDS,cap) for cap in CAPS}
assert np.array_equal(PLANS[10],FULL_PLANS[:DRAWS])
plan_rows = []
for cap,plans in PLANS.items():
    validate_plans(plans,PIDS,LABELS,cap)
    assert plans.shape[-1] > max(K_VALUES)+1
    for r,(C,I) in enumerate(plans):
        for side,ids in enumerate([C,I]):
            assert set(ids)<=set(FULL_PLANS[r,side])
        plan_rows.append(dict(cap=cap,draw=r,n_per_class=len(C),n_problems=len(np.unique(PIDS[C]))))
PLAN_SUMMARY = pd.DataFrame(plan_rows)
input_hashes = {str(p):digest(p) for p in [SOURCE13/'manifest.json',SOURCE13/'cohort_membership.csv',
    SOURCE13/'common_clean_plans.npz',SOURCE13/'aligned_id_summary.csv']}
# Cache hashes were verified above; do not re-read multi-GB representations.
for path in [ORIGINAL,PREFIX]:
    input_hashes[str(path)] = next(v for k,v in SOURCE_MANIFEST['inputs'].items() if Path(k).name==path.name)
MANIFEST = dict(config=CONFIG, inputs=input_hashes,
    versions=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,torch=torch.__version__),
    device=DEVICE, source13=str(SOURCE13), n_attempts=len(META), n_problems=len(np.unique(PIDS)),
    notes=['Exploratory follow-up motivated by existing results; not independent confirmation.',
           'Caps change both N and within-problem sampling composition.',
           'Detailed neighbors saved only for configured detail_draws; summaries use every draw.',
           'No permutation p-values or new LOPO inference in this diagnostic notebook.'])
SIGNATURE = hashlib.sha256(json.dumps(MANIFEST,sort_keys=True).encode()).hexdigest()
OUT = BASE/'results/neighborhood_diagnostic_phi2_v1'/('quick' if QUICK else 'final')/SIGNATURE[:12]
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'checkpoints').mkdir(exist_ok=True)
(OUT/'manifest.json').write_text(json.dumps(MANIFEST,indent=2))
META.to_csv(OUT/'cohort_keys.csv',index=False)
PLAN_SUMMARY.to_csv(OUT/'plan_summary.csv',index=False)
atomic_npz(OUT/'matched_plans.npz',signature=np.asarray(SIGNATURE),cohort_keys=KEYS,
           **{f'cap_{cap}':p for cap,p in PLANS.items()})
print('Results:',OUT)
display(PLAN_SUMMARY.groupby('cap')[['n_per_class','n_problems']].first())
print('Caps preserve every problem. N changes together with within-problem multiplicity.')


The source reproduction gate precedes each layer’s diagnostics. Checkpoints include compact all-draw summaries and raw neighbor/local arrays for the fixed detail draws.

In [ ]:
# 5 — Run/resume per representation and layer. A failed source reproduction blocks that layer.
def checkpoint_path(view,layer):
    return OUT/'checkpoints'/f'{view}__L{layer:02d}.npz'

@torch.no_grad()
def distance_matrix(raw):
    # Same NumPy float64 normalization and direct torch distance route as notebook 13.
    x = torch.from_numpy(normalize_rows(raw)).to(DEVICE)
    matrix = np.empty((len(x),len(x)),dtype=np.float64)
    for start in range(0,len(x),256):
        matrix[start:start+256] = torch.cdist(x[start:start+256],x,p=2,
            compute_mode='donot_use_mm_for_euclid_dist').cpu().numpy()
    np.fill_diagonal(matrix,np.inf)
    off = ~np.eye(len(matrix),dtype=bool)
    bad = off & ((matrix<=1e-12)|~np.isfinite(matrix))
    if bad.any():
        pairs = np.argwhere(np.triu(bad,1))[:20]
        report = pd.DataFrame([dict(a=int(a),b=int(b),problem_a=int(KEYS[a,0]),attempt_a=int(KEYS[a,1]),
            problem_b=int(KEYS[b,0]),attempt_b=int(KEYS[b,1]),distance=matrix[a,b]) for a,b in pairs])
        report.to_csv(OUT/'distance_failure_pairs.csv',index=False)
        raise FloatingPointError('Invalid/zero distances remain. See distance_failure_pairs.csv. No filtering applied.')
    if not np.allclose(matrix,matrix.T,rtol=1e-12,atol=1e-12):
        raise FloatingPointError('Distance matrix is not symmetric')
    return matrix

if not RUN_DIAGNOSTICS:
    raise RuntimeError('Audit complete. Set RUN_DIAGNOSTICS=True, then run cells 5–7.')
for view in VIEWS:
    pending = []
    for layer in LAYERS:
        path = checkpoint_path(view,layer)
        if path.exists():
            read_signed_npz(path,SIGNATURE)
        else: pending.append(layer)
    if not pending:
        print(view,': all checkpoints verified; no cache loading needed')
        continue
    # Load one trusted local extraction cache at a time, never the language model.
    path = ORIGINAL if view=='finals' else PREFIX
    cache = torch.load(path,map_location='cpu',weights_only=False)
    if not isinstance(cache,list): raise ValueError('Expected the original list-of-dicts cache')
    lookup = {(int(r['problem_id']),int(r['attempt_id'])):r for r in cache}
    assert len(lookup)==len(cache)
    records = [lookup[tuple(key)] for key in KEYS]
    for j,r in enumerate(records):
        assert bool_values([r['correct']])[0]==LABELS[j]
        assert tuple(r[view].shape)==(32,2560)
    # Retain only the chosen view, freeing other cached representations before distances.
    tensors = [r[view] for r in records]
    del records,lookup,cache;gc.collect()
    for layer in tqdm(pending,desc=f'{view}: layers'):
        raw = np.stack([t[layer].double().numpy() for t in tensors])
        D = distance_matrix(raw)
        del raw
        baseline = baseline_ids(D,FULL_PLANS)
        reproduction = []
        for j,k in enumerate([10,20]):
            ref = REFERENCE[(REFERENCE.representation==view)&(REFERENCE.layer==layer)&(REFERENCE.k==k)].iloc[0]
            actual = [baseline[:,0,j].mean(),baseline[:,1,j].mean(),(baseline[:,1,j]-baseline[:,0,j]).mean()]
            for metric,value in zip(['pooled_id_correct','pooled_id_incorrect','pooled_delta_id'],actual):
                passed = bool(np.isclose(value,float(ref[metric]),rtol=1e-6,atol=1e-8))
                reproduction.append(dict(representation=view,layer=layer,k=k,metric=metric,
                    reference=float(ref[metric]),recomputed=float(value),difference=float(value-ref[metric]),passed=passed))
        if not all(r['passed'] for r in reproduction):
            pd.DataFrame(reproduction).to_csv(OUT/f'reproduction_failure_{view}_L{layer:02d}.csv',index=False)
            raise RuntimeError('Original k=10/20 reproduction failed. Diagnose before interpreting new k values.')
        draws, ranks, shifts, details = [], [], [], {}
        for cap,plans in PLANS.items():
            for r,plan in enumerate(tqdm(plans,desc=f'L{layer:02d} cap={cap}',leave=False)):
                for side,indices in enumerate(plan):
                    context = dict(representation=view,layer=layer,cap=cap,draw=r,group=['C','I'][side])
                    try:
                        rows,rank_rows,shift_row,detail = diagnose_class(D,indices,PIDS,K_VALUES)
                    except (ValueError,FloatingPointError) as exc:
                        raise RuntimeError(f'{context}: {exc}') from exc
                    if cap==10:
                        for k,j in [(10,0),(20,1)]:
                            assert np.isclose(rows[K_VALUES.index(k)]['id_mean'],baseline[r,side,j],rtol=1e-12,atol=1e-12)
                    draws.extend([{**context,**row} for row in rows])
                    ranks.extend([{**context,**row} for row in rank_rows])
                    shifts.append({**context,**shift_row})
                    if r in DETAIL_DRAWS:
                        for name,a in detail.items(): details[f'cap{cap}_draw{r}_{context["group"]}_{name}'] = a
        # Rank summaries aggregate across draws; detailed fixed-draw distances remain in this checkpoint.
        rank_frame = pd.DataFrame(ranks)
        rank_summary = rank_frame.groupby(['representation','layer','cap','group','rank'],as_index=False)[
            ['distance_mean','distance_median','own_fraction','own_expected']].mean()
        arrays = dict(signature=np.asarray(SIGNATURE),
            draws_json=np.asarray(pd.DataFrame(draws).to_json(orient='records',double_precision=15)),
            ranks_json=np.asarray(rank_summary.to_json(orient='records',double_precision=15)),
            shifts_json=np.asarray(pd.DataFrame(shifts).to_json(orient='records',double_precision=15)),
            reproduction_json=np.asarray(pd.DataFrame(reproduction).to_json(orient='records',double_precision=15)),
            baseline=baseline,**details)
        atomic_npz(checkpoint_path(view,layer),**arrays)
        del D,arrays,details,draws,ranks,shifts,rank_frame,rank_summary;gc.collect()
    del tensors;gc.collect()
print('All requested checkpoints complete. Original k=10/20 reproduction passed for every computed layer.')


This section reads checkpoints. Missing layers stop the summary instead of silently producing a partial FINAL result.

In [ ]:
# 6 — Reload saved checkpoints, aggregate within draws first, then summarize across draws.
# This cell requires no representation loading or distance recomputation.
from io import StringIO
frames = {k:[] for k in ['draws','ranks','shifts','reproduction']}
for view in VIEWS:
    for layer in LAYERS:
        path = checkpoint_path(view,layer)
        if not path.exists(): raise RuntimeError(f'Missing checkpoint: {path}. Resume cell 5.')
        with np.load(path,allow_pickle=False) as z:
            if str(z['signature'].item())!=SIGNATURE: raise ValueError('Checkpoint signature mismatch')
            for key in frames: frames[key].append(pd.read_json(StringIO(str(z[key+'_json'].item()))))
DRAW = pd.concat(frames['draws'],ignore_index=True)
RANK = pd.concat(frames['ranks'],ignore_index=True)
SHIFT = pd.concat(frames['shifts'],ignore_index=True)
REPRO = pd.concat(frames['reproduction'],ignore_index=True)
assert REPRO.passed.all()
assert len(DRAW)==len(VIEWS)*len(LAYERS)*len(CAPS)*DRAWS*2*len(K_VALUES)
assert not DRAW.duplicated(['representation','layer','cap','draw','group','k']).any()
assert np.isfinite(DRAW.select_dtypes(include='number')).all().all()

# C and I use identical problem counts; paired comparisons retain the draw index.
PAIRED = DRAW.pivot(index=['representation','layer','cap','draw','k'],columns='group',values='id_mean').reset_index()
PAIRED.columns.name=None
PAIRED['delta']=PAIRED.I-PAIRED.C

def summarize_values(frame,keys,values):
    rows=[]
    for key,g in frame.groupby(keys,sort=True):
        if not isinstance(key,tuple):key=(key,)
        row=dict(zip(keys,key));row['draws']=int(len(g))
        for value in values:
            a=g[value].to_numpy(float)
            if not np.isfinite(a).all():raise FloatingPointError(f'Invalid {value}')
            row[value+'_mean']=float(a.mean())
            row[value+'_sampling_p025']=float(np.quantile(a,.025))
            row[value+'_sampling_p975']=float(np.quantile(a,.975))
        rows.append(row)
    return pd.DataFrame(rows)

SUMMARY = summarize_values(PAIRED,['representation','layer','cap','k'],['C','I','delta'])
# Each distribution statistic is computed inside a draw first. Avoid treating repeated attempts as independent.
metrics = [c for c in DRAW.columns if c not in ['representation','layer','cap','draw','group','k','n']]
LOCAL_SUMMARY = DRAW.groupby(['representation','layer','cap','group','k'],as_index=False)[metrics].mean()
SHIFT_SUMMARY = SHIFT.groupby(['representation','layer','cap','group'],as_index=False).mean(numeric_only=True).drop(columns='draw')
KSHIFT = PAIRED[PAIRED.k.isin([10,20])].pivot(index=['representation','layer','cap','draw'],columns='k',values=['C','I','delta'])
shift_draws = KSHIFT.index.to_frame(index=False)
for metric in ['C','I','delta']:
    shift_draws[metric+'_20_minus_10']=(KSHIFT[(metric,20)]-KSHIFT[(metric,10)]).to_numpy()
KSHIFT_SUMMARY = summarize_values(shift_draws,['representation','layer','cap'],['C_20_minus_10','I_20_minus_10','delta_20_minus_10'])
view_wide = PAIRED.pivot(index=['layer','cap','draw','k'],columns='representation',values='delta')
view_draws = view_wide.index.to_frame(index=False)
view_draws['pre_last_minus_finals']=(view_wide.pre_last-view_wide.finals).to_numpy()
VIEW_SUMMARY = summarize_values(view_draws,['layer','cap','k'],['pre_last_minus_finals'])

# Late-layer summary exists only when the complete fixed window is present.
HAVE_LATE = set(range(24,32)).issubset(LAYERS)
LATE = pd.DataFrame();LATE_KSHIFT=pd.DataFrame();LATE_VIEWS=pd.DataFrame()
if HAVE_LATE:
    late_draws = PAIRED[PAIRED.layer.between(24,31)].groupby(['representation','cap','draw','k'],as_index=False)[['C','I','delta']].mean()
    LATE = summarize_values(late_draws,['representation','cap','k'],['C','I','delta'])
    late_shift = shift_draws[shift_draws.layer.between(24,31)].groupby(['representation','cap','draw'],as_index=False)[['C_20_minus_10','I_20_minus_10','delta_20_minus_10']].mean()
    LATE_KSHIFT = summarize_values(late_shift,['representation','cap'],['C_20_minus_10','I_20_minus_10','delta_20_minus_10'])
    late_view = view_draws[view_draws.layer.between(24,31)].groupby(['cap','draw','k'],as_index=False).pre_last_minus_finals.mean()
    LATE_VIEWS = summarize_values(late_view,['cap','k'],['pre_last_minus_finals'])
else:
    print('QUICK: no L24–L31 aggregate. The complete fixed window has not been computed.')

# Descriptive paired sample-cap contrasts, not a pure sample-size causal effect.
cap_wide=PAIRED.pivot(index=['representation','layer','draw','k'],columns='cap',values='delta')
cap_frames=[]
for cap in [3,1]:
    f=cap_wide.index.to_frame(index=False);f['cap']=cap;f['delta_minus_cap10']=(cap_wide[cap]-cap_wide[10]).to_numpy();cap_frames.append(f)
CAP_CONTRASTS=summarize_values(pd.concat(cap_frames),['representation','layer','cap','k'],['delta_minus_cap10'])
exports = {'id_summary':SUMMARY,'local_distribution_summary':LOCAL_SUMMARY,'rank_summary':RANK,
    'local_shift_summary':SHIFT_SUMMARY,'k20_minus_k10_summary':KSHIFT_SUMMARY,
    'representation_contrasts':VIEW_SUMMARY,'sample_cap_contrasts':CAP_CONTRASTS,'reproduction_checks':REPRO,
    'paired_draw_metrics':PAIRED,'local_shift_draws':SHIFT}
if HAVE_LATE:exports.update(late_layer_summary=LATE,late_k20_minus_k10=LATE_KSHIFT,late_representation_contrasts=LATE_VIEWS)
for name,frame in exports.items():frame.to_csv(OUT/(name+'.csv'),index=False)
DRAW.to_csv(OUT/'local_distribution_by_draw.csv',index=False)
print('Reference reproduction:',len(REPRO),'checks passed')
display((LATE if HAVE_LATE else SUMMARY[SUMMARY.layer==31]).query('cap==10')[['representation','k','C_mean','I_mean','delta_mean']].round(4))
display((LATE_KSHIFT if HAVE_LATE else KSHIFT_SUMMARY[KSHIFT_SUMMARY.layer==31]).query('cap==10').round(4))
print('Sampling percentiles describe matched-draw variability. They are not confidence intervals.')


Figures retain all requested layers. The compact review bundle excludes large raw checkpoints; keep the entire run directory in Drive.

In [ ]:
# 7 — Figures, raw-detail inspection example, and compact review bundle.
FIG = OUT/'figures';FIG.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi':120,'font.size':10})
for metric,label in [('C','ID(correct)'),('I','ID(incorrect)'),('delta','Delta ID (incorrect - correct)')]:
    fig,axes=plt.subplots(1,2,figsize=(13,4),sharey=True)
    for ax,view in zip(axes,VIEWS):
        for k in K_VALUES:
            g=SUMMARY[(SUMMARY.representation==view)&(SUMMARY.cap==10)&(SUMMARY.k==k)].sort_values('layer')
            ax.plot(g.layer,g[metric+'_mean'],label=f'k={k}',marker='.' if QUICK else None)
        ax.set(title=view,xlabel='Layer',ylabel=label)
        if metric=='delta':ax.axhline(0,color='gray',lw=.7)
        ax.legend(ncol=2,fontsize=8)
    fig.suptitle(('QUICK: selected layers, execution test' if QUICK else 'All layers | original matched plans'))
    fig.tight_layout();fig.savefig(FIG/f'all_layers_{metric}.png');plt.show();plt.close(fig)

# k-curves: full late window in FINAL; explicitly L31 only in QUICK.
curve=LATE if HAVE_LATE else SUMMARY[SUMMARY.layer==31]
window='L24–L31' if HAVE_LATE else 'L31 only (QUICK)'
fig,axes=plt.subplots(2,3,figsize=(15,8))
for row,view in enumerate(VIEWS):
    for col,metric in enumerate(['C','I','delta']):
        ax=axes[row,col]
        for cap in CAPS:
            g=curve[(curve.representation==view)&(curve.cap==cap)].sort_values('k')
            n=int(PLAN_SUMMARY[PLAN_SUMMARY.cap==cap].n_per_class.iloc[0])
            ax.plot(g.k,g[metric+'_mean'],marker='o',label=f'cap={cap}, N={n}/class')
            ax.fill_between(g.k.to_numpy(),g[metric+'_sampling_p025'].to_numpy(),g[metric+'_sampling_p975'].to_numpy(),alpha=.12)
        ax.set(title=f'{view}: {metric}',xlabel='k',ylabel='Estimated ID' if metric!='delta' else 'Delta ID')
        ax.legend(fontsize=8)
fig.suptitle(window+' | bands: matched-draw percentiles, not confidence intervals')
fig.tight_layout();fig.savefig(FIG/'k_and_sample_caps.png');plt.show();plt.close(fig)

# Rank composition and distance by every layer; no layer chosen because of a striking result.
for value,label in [('own_fraction','Fraction of neighbors from the same problem'),('distance_mean','Mean neighbor distance')]:
    fig,axes=plt.subplots(2,2,figsize=(12,8))
    for row,view in enumerate(VIEWS):
        for col,group in enumerate(['C','I']):
            g=RANK[(RANK.representation==view)&(RANK.cap==10)&(RANK.group==group)]
            grid=g.pivot(index='layer',columns='rank',values=value).sort_index()
            opts=dict(vmin=0,vmax=1) if value=='own_fraction' else {}
            im=axes[row,col].imshow(grid.to_numpy(),aspect='auto',origin='lower',**opts)
            axes[row,col].set(title=f'{view} / {group}',xlabel='Neighbor rank',ylabel='Layer')
            ticks=[0,9,19,29,39];axes[row,col].set_xticks(ticks,[1,10,20,30,40])
            ys=list(range(len(grid))) if QUICK else list(range(0,len(grid),4))+[len(grid)-1]
            axes[row,col].set_yticks(ys,[int(grid.index[j]) for j in ys])
            fig.colorbar(im,ax=axes[row,col])
    fig.suptitle(label+' | full sample, averages over draws')
    fig.tight_layout();fig.savefig(FIG/(value+'_heatmap.png'));plt.show();plt.close(fig)

local_window=LOCAL_SUMMARY[LOCAL_SUMMARY.layer.between(24,31)] if HAVE_LATE else LOCAL_SUMMARY[LOCAL_SUMMARY.layer==31]
local_window=local_window.groupby(['representation','cap','group','k'],as_index=False).mean(numeric_only=True)
fig,axes=plt.subplots(1,2,figsize=(13,4))
for ax,view in zip(axes,VIEWS):
    for group in ['C','I']:
        g=local_window[(local_window.representation==view)&(local_window.cap==10)&(local_window.group==group)].sort_values('k')
        ax.plot(g.k,g.id_mean,label=f'{group}: mean')
        ax.plot(g.k,g.local_median,ls='--',label=f'{group}: median')
        ax.plot(g.k,g.local_p95,ls=':',label=f'{group}: 95th percentile')
    ax.set(title=view,xlabel='k',ylabel='Local ID distribution summaries');ax.legend(fontsize=8)
fig.suptitle(window+' | distribution statistics averaged over draws and layers')
fig.tight_layout();fig.savefig(FIG/'local_id_distributions.png');plt.show();plt.close(fig)

# One fixed-draw empirical distribution, explicitly exploratory; numerical distributions above use all draws.
fig,axes=plt.subplots(1,2,figsize=(13,4))
for ax,view in zip(axes,VIEWS):
    with np.load(checkpoint_path(view,31),allow_pickle=False) as z:
        for group in ['C','I']:
            a=z[f'cap10_draw0_{group}_local_id']
            for k in [10,20]:
                vals=np.sort(a[:,K_VALUES.index(k)])
                ax.step(vals,np.arange(1,len(vals)+1)/len(vals),where='post',label=f'{group}, k={k}')
    ax.set(title=view,xlabel='Local ID',ylabel='Empirical cumulative fraction',xscale='log');ax.legend(fontsize=8)
fig.suptitle('L31, original matched draw 0: fixed diagnostic example, not independent observations')
fig.tight_layout();fig.savefig(FIG/'local_id_ecdf_draw0_L31.png');plt.show();plt.close(fig)

# Inspect actual neighbor identities from any saved detail draw. Index values refer to cohort_keys.csv.
def neighbor_table(view,layer,cap=10,draw=0,group='C',query_position=0):
    if draw not in DETAIL_DRAWS:raise ValueError('This draw has summaries only; choose a saved detail draw')
    stem=f'cap{cap}_draw{draw}_{group}_'
    with np.load(checkpoint_path(view,layer),allow_pickle=False) as z:
        q=int(z[stem+'query_indices'][query_position]);n=z[stem+'neighbor_indices'][query_position]
        distances=z[stem+'distances'][query_position]
    table=META.iloc[n][['problem_id','attempt_id']].reset_index(drop=True)
    table.insert(0,'rank',np.arange(1,len(n)+1));table['distance']=distances
    table['same_problem']=table.problem_id.to_numpy()==PIDS[q]
    table['query_problem']=int(KEYS[q,0]);table['query_attempt']=int(KEYS[q,1])
    return table
example=neighbor_table('pre_last',31)
example.to_csv(OUT/'neighbor_example_pre_last_L31_draw0_C_query0.csv',index=False)
display(example.head(20))

notes='''# Neighborhood-size diagnostic review

This is an exploratory follow-up to the existing aligned analysis. Read manifest.json for QUICK/FINAL.
Original FINAL 13 k=10/20 class means and Delta ID are reproduced with all 120 original plans before each layer's diagnostic runs. Quick diagnostics use only the first 3 plans and layers 0,24,31; no late-window aggregate is produced in QUICK.

Primary estimator is unchanged: mean of local Levina–Bickel inverse estimates, k-1 denominator, row L2 normalization, float64 direct distances. It is not the inverse of a pooled mean logarithm. No coordinate standardization, duplicate removal, jitter, clipping, trimming or replacement resampling is added.

All k values and both views use identical attempts in a draw. Caps 3 and 1 use nested prefixes of each original per-problem random selection and retain every problem. They change sample size AND within-problem multiplicity; they do not isolate a pure N effect. Equal k across views/classes/caps does not imply equal physical radius.

Sampling percentiles are over matched draws, not confidence intervals or independent replications. Late summaries average layers inside each draw first. Local quantiles and top-tail shares are computed inside each draw, then averaged; do not treat their rows as independent observations. No permutation or LOPO analysis is newly performed here.

local_shift_summary: local k20-k10 changes on the same query points. Positive plus negative contribution equals the mean shift. top_abs5_contribution plus remainder_contribution also equals the mean shift; these are diagnostic decompositions, not a trimmed replacement estimator. top_abs5_share is a share of absolute changes, not a causal share.

local_distribution_summary: top5_share is the share of the sum of local IDs contributed by the largest ceil(0.05*N) local estimates. own_expected uses the sampled same-problem count divided by N-1, conditional on each query. Ties are resolved by original plan order; boundary tie fractions reveal ambiguity in neighbor identities.

rank_summary averages rank-specific distances and same-problem membership over draws. Detailed neighbor IDs/distances and local IDs for fixed draw 0 (all computed layers/caps/classes) remain in per-layer checkpoints, indexed by cohort_keys.csv. These large checkpoints are intentionally excluded from this compact review bundle. local_distribution_by_draw.csv remains in the run directory too.

k=5 and k=8 are explicit small-neighborhood diagnostics. No automatic k selection. Stable behavior does not establish a causal reasoning mechanism. Cached extraction precision is inherited: float64 distance computation cannot restore precision lost during extraction.
'''
(OUT/'REVIEW_NOTES.md').write_text(notes)
review_files=[OUT/'manifest.json',OUT/'REVIEW_NOTES.md',OUT/'cohort_keys.csv',OUT/'plan_summary.csv',
              OUT/'matched_plans.npz',OUT/'neighbor_example_pre_last_L31_draw0_C_query0.csv']
review_files += [OUT/(name+'.csv') for name in exports]
review_files += sorted(FIG.glob('*.png'))
with zipfile.ZipFile(OUT/'review_bundle.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in review_files:z.write(path,path.relative_to(OUT))
print('Mode:', 'QUICK execution test' if QUICK else 'FINAL diagnostic run')
print('Review bundle:',OUT/'review_bundle.zip')
print('Keep this run directory: it contains checkpoints and detailed neighbor records.')


## Reading the result without overclaiming

- If many local k20-minus-k10 changes have the same sign, the mean shift is distributed across points. If a few large changes dominate, inspect their neighborhoods before claiming a broad geometric transition.
- If same-problem composition changes across neighbor ranks, that supports a clustering explanation worth testing. It does not establish that clustering caused the ID change.
- Cap=1 guarantees zero same-problem neighbors. A changed effect under this cap is ambiguous between sample size, within-problem multiplicity, and changed problem weighting.
- Strong dependence on neighborhood radius or sample cap limits interpretation as one fixed intrinsic dimension. Do not select a favorable k or replace the primary estimator with a median/trimmed mean after seeing results.
- Exact duplicate filtering does not remove all near-duplicates or restore precision lost in stored representations. Float64 calculations only address calculation precision.

Estimator reference: [Levina and Bickel, Maximum Likelihood Estimation of Intrinsic Dimension](https://papers.neurips.cc/paper/2577-maximum-likelihood-estimation-of-intrinsic-dimension.pdf), especially equation (8) and the discussion of neighborhood size. The established cohort selection and hypotheses already used these data, so this follow-up is exploratory.